In [2]:
#create dataframe
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "customer": ["Ali", "Sara", "Reza", "Neda", "Hassan"],
    "age": [25, np.nan, 31, 28, np.nan],
    "city": ["Tehran", "Shiraz", np.nan, "Tabriz", "Tehran"],
    "income": [1200, 1800, 1500, np.nan, 2100]
})

df

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,NaN,Shiraz,1800.0
2,Reza,31.0,NaN,1500.0
3,Neda,28.0,Tabriz,NaN
4,Hassan,NaN,Tehran,2100.0


In [3]:
#simplest way to fill is fill with a fixed value
df.fillna(0)

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,0.0,Shiraz,1800.0
2,Reza,31.0,0,1500.0
3,Neda,28.0,Tabriz,0.0
4,Hassan,0.0,Tehran,2100.0


This replaces every missing value in the DataFrame with zero.

Technically, that works. But is it good data cleaning?
Not necessarily. An age of zero means something different from an unknown age, and a city of zero makes little sense.

In [5]:
#fill missing values with fixed value
#1- Replace missing values in one column
df['age'] = df['age'].fillna(0)
df

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,0.0,Shiraz,1800.0
2,Reza,31.0,NaN,1500.0
3,Neda,28.0,Tabriz,NaN
4,Hassan,0.0,Tehran,2100.0


In [6]:
#But for a categorical column such as city, a better option might be "Unknown":
df['city'] = df['city'].fillna('Unknown')
df

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,0.0,Shiraz,1800.0
2,Reza,31.0,Unknown,1500.0
3,Neda,28.0,Tabriz,NaN
4,Hassan,0.0,Tehran,2100.0


In [7]:
#2- Fill different columns with different values
df = pd.DataFrame({
    "customer": ["Ali", "Sara", "Reza", "Neda", "Hassan"],
    "age": [25, np.nan, 31, 28, np.nan],
    "city": ["Tehran", "Shiraz", np.nan, "Tabriz", "Tehran"],
    "income": [1200, 1800, 1500, np.nan, 2100]
})

df_clean = df.fillna({
    'age':0,
    'city':'Unknown',
    'income':0
})

df_clean

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,0.0,Shiraz,1800.0
2,Reza,31.0,Unknown,1500.0
3,Neda,28.0,Tabriz,0.0
4,Hassan,0.0,Tehran,2100.0


In [9]:
df_clean.isna().sum()

customer    0
age         0
city        0
income      0
dtype: int64

In [10]:
#3- Mean, median, and mode
#Fill with the mean
mean_age = df['age'].mean()
df['age'] = df['age'].fillna(mean_age)
df

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,28.0,Shiraz,1800.0
2,Reza,31.0,NaN,1500.0
3,Neda,28.0,Tabriz,NaN
4,Hassan,28.0,Tehran,2100.0


Pandas' mean() skips missing values by default.

In [11]:
#Fill with the median
median_income = df['income'].median()
df['income'] = df['income'].fillna(median_income)
df

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,28.0,Shiraz,1800.0
2,Reza,31.0,NaN,1500.0
3,Neda,28.0,Tabriz,1650.0
4,Hassan,28.0,Tehran,2100.0


In [12]:
#Fill with the mode
#The mode is the most frequent value. It is useful for categorical columns.
most_common_city = df['city'].mode()[0]
df['city'] = df['city'].fillna(most_common_city)
df

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,28.0,Shiraz,1800.0
2,Reza,31.0,Tehran,1500.0
3,Neda,28.0,Tabriz,1650.0
4,Hassan,28.0,Tehran,2100.0


Why [0]? Because mode() returns a Series, and there can be more than one mode if multiple values tie for the highest frequency.

In [17]:
#A cleaner, reusable pattern
df = pd.DataFrame({
    "customer": ["Ali", "Sara", "Reza", "Neda", "Hassan"],
    "age": [25, np.nan, 31, 28, np.nan],
    "city": ["Tehran", "Shiraz", np.nan, "Tabriz", "Tehran"],
    "income": [1200, 1800, 1500, np.nan, 2100]
})

df_clean = df.copy()

df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean['city'] = df_clean['city'].fillna('Unknown')
df_clean['income'] = df_clean['income'].fillna(df['income'].median())

In [18]:
df

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,NaN,Shiraz,1800.0
2,Reza,31.0,NaN,1500.0
3,Neda,28.0,Tabriz,NaN
4,Hassan,NaN,Tehran,2100.0


In [19]:
df_clean

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,28.0,Shiraz,1800.0
2,Reza,31.0,Unknown,1500.0
3,Neda,28.0,Tabriz,1650.0
4,Hassan,28.0,Tehran,2100.0


In [20]:
#Forward fill and backward fill
#Sometimes, the order of the data matters. This is particularly common with dates, measurements, and time series.
daily = pd.DataFrame({
    "date": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"],
    "temperature": [20, np.nan, 22, np.nan, 24]
})

daily

,date,temperature
0,Monday,20.0
1,Tuesday,NaN
2,Wednesday,22.0
3,Thursday,NaN
4,Friday,24.0


In [24]:
#ffill() : Forward fill uses the most recent preceding non-missing value.
daily['temperature'] = daily['temperature'].ffill()
daily

,date,temperature
0,Monday,20.0
1,Tuesday,20.0
2,Wednesday,22.0
3,Thursday,22.0
4,Friday,24.0


This can make sense when a value is expected to remain valid until a new measurement arrives, such as a recorded account status.
However, it would not always be reasonable to assume that an unrecorded temperature is identical to the previous day's temperature.

In [32]:
#bfill() : Backward fill uses the next available non-missing value.
daily = pd.DataFrame({
    "date": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"],
    "temperature": [20, np.nan, 22, np.nan, 24]
})
daily['temperature'] = daily['temperature'].bfill()
daily

,date,temperature
0,Monday,20.0
1,Tuesday,22.0
2,Wednesday,22.0
3,Thursday,24.0
4,Friday,24.0


This can be useful in certain ordered datasets, but it uses future observations to fill earlier gaps. That may be inappropriate when simulating what was known at a particular point in time.

In [33]:
#Limit how far you fill
daily["temperature"].ffill(limit=1)
#This fills at most one consecutive missing value at a time.

0    20.0
1    22.0
2    22.0
3    24.0
4    24.0
Name: temperature, dtype: float64

In [34]:
#Practice and analyst challenge
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer": ["Ali", "Sara", "Reza", "Neda", "Hassan", "Maryam"],
    "age": [25, np.nan, 31, 28, np.nan, 35],
    "city": ["Tehran", "Shiraz", np.nan, "Tabriz", "Tehran", np.nan],
    "income": [1200, 1800, 1500, np.nan, 2100, 2400]
})

In [35]:
customers

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,NaN,Shiraz,1800.0
2,Reza,31.0,NaN,1500.0
3,Neda,28.0,Tabriz,NaN
4,Hassan,NaN,Tehran,2100.0
5,Maryam,35.0,NaN,2400.0


In [37]:
#Replace missing values in city with "Unknown".
customers['city'] = customers['city'].fillna('Unknown')
customers

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,NaN,Shiraz,1800.0
2,Reza,31.0,Unknown,1500.0
3,Neda,28.0,Tabriz,NaN
4,Hassan,NaN,Tehran,2100.0
5,Maryam,35.0,Unknown,2400.0


In [38]:
#Replace missing ages with the median age.
customers['age'] = customers['age'].fillna(customers['age'].median())
customers

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,29.5,Shiraz,1800.0
2,Reza,31.0,Unknown,1500.0
3,Neda,28.0,Tabriz,NaN
4,Hassan,29.5,Tehran,2100.0
5,Maryam,35.0,Unknown,2400.0


In [39]:
#Replace missing incomes with the mean income.
customers['income'] = customers['income'].fillna(customers['income'].mean())
customers

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,29.5,Shiraz,1800.0
2,Reza,31.0,Unknown,1500.0
3,Neda,28.0,Tabriz,1800.0
4,Hassan,29.5,Tehran,2100.0
5,Maryam,35.0,Unknown,2400.0


In [40]:
customers = pd.DataFrame({
    "customer": ["Ali", "Sara", "Reza", "Neda", "Hassan", "Maryam"],
    "age": [25, np.nan, 31, 28, np.nan, 35],
    "city": ["Tehran", "Shiraz", np.nan, "Tabriz", "Tehran", np.nan],
    "income": [1200, 1800, 1500, np.nan, 2100, 2400]
})
#Create a new DataFrame called customers_clean that fills missing ages and incomes with their respective medians, 
#and missing cities with "Unknown". Preserve the original.
customers_clean = customers.copy()
customers_clean['age'] = customers_clean['age'].fillna(customers_clean['age'].median())
customers_clean['income'] = customers_clean['income'].fillna(customers_clean['income'].median())
customers_clean['city'] = customers_clean['city'].fillna('Unknown')
customers_clean

,customer,age,city,income
0,Ali,25.0,Tehran,1200.0
1,Sara,29.5,Shiraz,1800.0
2,Reza,31.0,Unknown,1500.0
3,Neda,28.0,Tabriz,1800.0
4,Hassan,29.5,Tehran,2100.0
5,Maryam,35.0,Unknown,2400.0


In [41]:
#Using the daily DataFrame from Section 4, fill missing temperatures with forward fill, 
#allowing at most one consecutive missing value to be filled.
daily = pd.DataFrame({
    "date": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"],
    "temperature": [20, np.nan, 22, np.nan, 24]
})
daily['temperature'] = daily['temperature'].ffill(limit=1)
daily

,date,temperature
0,Monday,20.0
1,Tuesday,20.0
2,Wednesday,22.0
3,Thursday,22.0
4,Friday,24.0
